# 4-1 DICOMシリーズの読み込みと表示

範囲4 DICOMビューアの自作｜第9回

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/Medical_Imaging_Seminar_Public/blob/main/hiroshima/4_dicom_viewer/4-1_load_series.ipynb)

## この回で分かるようになること

- フォルダ内の複数のDICOMファイルを読み込み、シリーズごとに分けられる
- スライスの順番を、ヘッダの位置の情報で並べ替え、3次元の配列にまとめられる
- 読み込めないファイルが混ざっていても止まらない読み込み処理を作れる

## 使うデータ

- 練習用のフォルダ `data/viewer_sample`（2-1で実行する）。次のファイルを1つのフォルダに混ぜて作ります
  - 腹部CT（152枚）、腹部MRIの横断像（26枚）と矢状断像（26枚）、腹部X線写真（1枚、拡張子なし）：TCIA Pseudo-PHI-DICOM-Data（匿名化版、CC BY 4.0）
  - 超音波画像（1枚、JPEG 2000で圧縮されたDICOM）：pydicom付属のサンプル
  - DICOMではないテキストファイル（1つ）

> ここで作る関数は、第10回のビューア（4-2）でそのまま使います。

## 0. 準備

**Colab で開いた場合**: 上の「Open In Colab」ボタンから開き、次のセル（Colab用の準備）を実行します。`pydicom` と `idc-index` の追加インストールと、データ取得に使う `course_data.py` の取得を行います。1〜2分かかります。

**VS Code で開いた場合**: ノートブック右上の「カーネルの選択」で `.venv` を選びます。Colab用の準備セルは、そのまま実行しても何も起きません（`uv sync` で環境が整っているため）。

In [ ]:
import sys
import subprocess
import urllib.request

# Colab では、足りないライブラリと course_data.py（データ取得用）を用意する
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydicom", "idc-index"], check=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/course_data.py", "course_data.py"
    )
    print("準備できました")


In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from pydicom import examples
from pydicom.errors import InvalidDicomError
from course_data import fetch

SAMPLE_DIR = Path("data/viewer_sample")

## 1. 解説

### 1-1. DICOMビューアが内部でしていること

2-2のノートブック（1-1）で、ビューアが裏で行っている4つの処理を挙げました。

1. フォルダ内のファイルを読み込み、シリーズごとに分ける
2. スライスを位置の順に並べる
3. 画素値をCT値に換算する
4. CT値を、画面に表示する明るさに変換する

3と4は2-2で作りました。この回では1と2を作り、4つをつなげて、フォルダを指定するとシリーズの画像が表示されるところまで進みます。4-2では、これらの関数を画面の部品（スライダーなど）とつないでアプリにします。

### 1-2. スライスの並べ方

スライスを正しい順に並べるには、何を手がかりにすればよいでしょうか。候補は3つあります。

| 手がかり | 使えるか |
|---|---|
| ファイル名 | 使えない。この講座のデータのファイル名は、ランダムな文字列です。病院のシステムから書き出したファイルも、連番とは限りません |
| InstanceNumber | 目安にはなるが、頼れない。番号が頭側と足側のどちらから始まるかは決まっておらず、この回のCTでは1番が最も頭側です。番号がないファイルもあります |
| ImagePositionPatient | 使える。各スライスの左上の画素が、患者の体のどこにあるか（mm）を表します |

ImagePositionPatient は (x, y, z) の3つの値で、x は患者の右から左、y は前から後ろ、z は足から頭の向きに増えます。横断像ならスライスごとに z が変わるので、z で並べれば済みます。

ところが、矢状断像ではスライスごとに変わるのは x で、z で並べても正しい順になりません。どの向きの断面にも使える方法は、ImageOrientationPatient（画像の横方向と縦方向の向きを表す2つのベクトル）の外積でスライスに垂直な向き $\mathbf{n}$ を求め、各スライスの位置 $\mathbf{p}$ との内積 $\mathbf{p} \cdot \mathbf{n}$ で並べる方法です。

### 1-3. シリーズの区別

1つのフォルダに、複数のシリーズのファイルが混ざっていることはよくあります。検査のCDを開くと、CTの造影前と造影後、位置決め用の画像などが同じフォルダに入っていることもあります。

2-1で見たとおり、シリーズは SeriesInstanceUID で区別します。ただし、UIDは長い数字の列なので、ビューアで一覧にするときは、Modality、SeriesDescription、枚数を並べて表示するのが普通です。

### 1-4. 読み込めないファイル

実際のフォルダには、読み込めないファイルや、読み込み方に注意が必要なファイルが混ざります。

- **DICOMではないファイル**：説明書きのテキストや、ビューアのソフト本体など。`pydicom.dcmread` は `InvalidDicomError` を出す
- **拡張子のないDICOMファイル**：`.dcm` の付かないDICOMは珍しくない。拡張子で選ぶと見落とす
- **圧縮されたDICOM**：JPEGなどで圧縮された画素データは、環境によっては追加のライブラリがないと取り出せない
- **1枚だけのシリーズ**：X線写真や超音波画像。3次元の配列にする必要はない

ビューアは、読み込めないファイルがあっても止まらずに、読み飛ばしたファイルを知らせるように作ります。

## 2. 動かしてみる

教員が用意したコードを、上から順に実行します。

### 2-1. 練習用のフォルダを作る

データをダウンロードし、1つのフォルダに混ぜてコピーします。X線写真は拡張子を外した名前でコピーします。

In [ ]:
def make_sample_folder():
    """複数のシリーズと、DICOMではないファイルが混ざった練習用のフォルダを作る"""
    if SAMPLE_DIR.exists():
        shutil.rmtree(SAMPLE_DIR)
    SAMPLE_DIR.mkdir(parents=True)
    for name in ["ct", "mr", "mr_sag"]:
        for f in fetch(name).glob("*.dcm"):
            shutil.copy(f, SAMPLE_DIR / f.name)
    cr_file = next(fetch("cr").glob("*.dcm"))
    shutil.copy(cr_file, SAMPLE_DIR / "IMG0001")  # 拡張子のないDICOM
    shutil.copy(examples.get_path("jpeg2k"), SAMPLE_DIR / "us_jpeg2000.dcm")  # 圧縮されたDICOM
    (SAMPLE_DIR / "readme.txt").write_text("このファイルはDICOMではありません。\n", encoding="utf-8")


make_sample_folder()
print("作成しました:", SAMPLE_DIR)

### 2-2. フォルダ内のファイルを一覧にする

ファイル名と拡張子から、何の画像かが分かるかどうかを見ます。

In [ ]:
files = sorted(p for p in SAMPLE_DIR.iterdir() if p.is_file())
listing = pd.DataFrame({
    "name": [p.name for p in files],
    "suffix": [p.suffix or "(なし)" for p in files],
    "size (KB)": [round(p.stat().st_size / 1024) for p in files],
})
print("ファイルの数:", len(files))
print(listing["suffix"].value_counts())
listing.head(8)

### 2-3. 1枚を読み込んで表示する

CTのファイルを1つ探して読み込み、2-2で作った関数と同じ処理で表示します。

In [ ]:
def to_hu(ds):
    """画素値を実際の値（CTならCT値）に換算する。係数がないときはそのままの値を使う"""
    slope = float(ds.get("RescaleSlope", 1))
    intercept = float(ds.get("RescaleIntercept", 0))
    return ds.pixel_array.astype(np.float32) * slope + intercept


def apply_window(image, level, width):
    """ウィンドウの範囲に合わせて 0（黒）〜 1（白）の明るさに変換する"""
    low, high = level - width / 2, level + width / 2
    return np.clip((image - low) / (high - low), 0, 1)


for path in files:
    if path.suffix == ".dcm" and pydicom.dcmread(path, stop_before_pixels=True).Modality == "CT":
        ds = pydicom.dcmread(path)
        break

print("ファイル:", path.name)
print("SeriesDescription:", ds.SeriesDescription, " InstanceNumber:", ds.InstanceNumber,
      " ImagePositionPatient:", [float(v) for v in ds.ImagePositionPatient])

plt.figure(figsize=(5, 5))
plt.imshow(apply_window(to_hu(ds), 40, 400), cmap="gray", vmin=0, vmax=1)
plt.title(f"InstanceNumber {ds.InstanceNumber}")
plt.axis("off")
plt.show()

## 3. AIに頼んでみる

頼む前に、ソース管理（`Ctrl+Shift+G`）でここまでの状態をコミットします。AIが変更した後は、差分を見て採用するかどうかを決めます。頼み方は `setup/03_ai_assistant.md` を参照してください。

この回の基本課題1〜3で作る関数は、4-2のビューアで使います。関数の名前と、何を受け取って何を返すかを、課題に書いたとおりにしてください。発展課題は、基本課題を終えた人が取り組みます。

### 基本課題1：シリーズごとに数える

次の関数を作るよう頼みます。

- 名前：`scan_folder(folder)`
- 受け取るもの：フォルダのパス
- 返すもの：シリーズの一覧（SeriesInstanceUID ごとに、Modality、SeriesDescription、ファイルのパスのリスト）と、読み飛ばしたファイルのリスト
- 条件：拡張子で選ばず、フォルダ内のすべてのファイルを試す。DICOMでないファイルは読み飛ばす。画素データは読まない（`stop_before_pixels=True`）

作った関数で `SAMPLE_DIR` を調べ、シリーズごとの Modality、SeriesDescription、枚数を表にします。

### 基本課題2：スライスを並べ替えて3次元の配列にする

次の関数を作るよう頼みます。

- 名前：`load_volume(files)`
- 受け取るもの：1つのシリーズのファイルのパスのリスト
- 返すもの：スライスの位置の順に並べた3次元の配列（2-3の `to_hu` で換算した値）と、並べた順の各スライスの位置（1-2の $\mathbf{p} \cdot \mathbf{n}$）のリスト
- 条件：1-2の方法（スライスに垂直な向きへの位置）で並べる

CTとMRIの矢状断像の両方で試し、位置のリストが一方向に順に並んでいるか確かめます。

### 基本課題3：指定したスライスを表示する関数を作る

次の関数を作るよう頼みます。

- 名前：`show_slice(volume, index, level, width)`
- 受け取るもの：基本課題2の3次元の配列、スライスの番号、ウィンドウレベル、ウィンドウ幅
- すること：`apply_window` で明るさに変換して表示する

CTの最初のスライス、真ん中のスライス、最後のスライスを軟部組織のウィンドウで並べて表示し、頭と足のどちら側から並んでいるかを確かめます。

### 基本課題4：誤りを含むコードを読む

次のセルは、CTをファイル名の順に並べて3次元の配列を作り、体の中央で縦に切った断面（矢状断）を表示しています。表示された画像で何が起きているかを説明し、基本課題2の関数を使った場合と並べて比べます。

In [ ]:
# 教員が用意した、誤りを含むコード
ct_files = sorted(fetch("ct").glob("*.dcm"))  # ファイル名の順に並べている
volume_by_name = np.stack([to_hu(pydicom.dcmread(f)) for f in ct_files])

plt.figure(figsize=(5, 6))
plt.imshow(apply_window(volume_by_name[:, :, 256], 40, 400), cmap="gray", aspect=2.5 / 0.74)
plt.title("sagittal view?")
plt.axis("off")
plt.show()

### 発展課題1：冠状断と矢状断を表示する

CTの3次元の配列から、冠状断（体を前後に分ける断面）と矢状断（体を左右に分ける断面）を切り出して表示します。画素の間隔（PixelSpacing）とスライスの間隔が違うので、`aspect` を使って縦横の比を合わせ、頭が上になるように表示します。

## 4. AIの答えを確かめる

確認できた項目は `[ ]` を `[x]` に書き換えます。

- [ ] 基本課題1で、5つのシリーズ（CT 152枚、MRI横断像 26枚、MRI矢状断像 26枚、X線 1枚、超音波 1枚）が見つかり、`readme.txt` だけが読み飛ばされた
- [ ] 基本課題1で、拡張子のない `IMG0001` もX線のシリーズとして数えられている
- [ ] 基本課題2で、CTとMRIの矢状断像の両方について、並べた後の位置が一方向に順に並んでいる
- [ ] 基本課題3で、最初と最後のスライスを表示し、頭側と足側のどちらから並んでいるかを確かめた
- [ ] InstanceNumber の順番と、位置で並べた順番の関係を確かめた
- [ ] CTの枚数が、3D Slicerで表示される枚数と一致した

## 5. 振り返り

| 項目 | 記入欄 |
|---|---|
| 使ったプロンプト | |
| AIの答えで直した点・採用しなかった点 | |
| この回で分かったこと | |
| まだ分からないこと | |

記入したら保存してコミットします。提出のしかたは授業で指示します。